In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')
DATA_DIR = Path('..') / 'data'
files = {
  'SierraLeone': DATA_DIR / 'sierraleone_clean.csv',
dfs = {}
for name, path in files.items():
    if path.exists():
        dfs[name] = pd.read_csv(path)
        print(f'Loaded {name}:', dfs[name].shape)
    else:
        print(f'Missing cleaned CSV for {name}:', path)

In [ ]:
# Prepare a combined DataFrame with a standard set of columns (GHI,DNI,DHI) where available
def find_col(df, candidates):
    cols = {c.lower(): next((col for col in df.columns if col.lower().startswith(c)), None) for c in candidates}
    return cols
candidates = ['ghi','dni','dhi']
combined = []
for name, df in dfs.items():
    mapping = find_col(df, candidates)
    row = df.rename(columns={v:k.upper() for k,v in mapping.items() if v is not None})
    # keep only the standardized columns if present
    keep = [c for c in ['GHI','DNI','DHI'] if c in row.columns]
    if keep:
        tmp = row[keep].copy()
        tmp['country'] = name
        combined.append(tmp)
,

In [ ]:
# Boxplots for GHI, DNI, DHI side-by-side
metrics = [m for m in ['GHI','DNI','DHI'] if m in df_all.columns]
import math
n = len(metrics)
fig, axes = plt.subplots(1, max(1,n), figsize=(6*max(1,n),4))
if n == 1:
    axes = [axes]
for ax, metric in zip(axes, metrics):
    sns.boxplot(x='country', y=metric, data=df_all, ax=ax)
    ax.set_title(f'Boxplot {metric} by country')
plt.tight_layout()
plt.show()

In [ ]:
# Summary table: mean, median, std grouped by country for each metric
summary = df_all.groupby('country')[metrics].agg(['mean','median','std']).round(3)
,

In [ ]:
# Statistical testing: Kruskal-Wallis (non-parametric) if SciPy is available
try:
    from scipy.stats import kruskal
    print('Running Kruskal-Wallis tests (GHI/DNI/DHI)')
    for metric in metrics:
        groups = [grp[metric].dropna().values for name, grp in df_all.groupby('country')]
        stat, p = kruskal(*groups)
        print(f'{metric}: H={stat:.3f}, p={p:.4g}')
except Exception as e:
    print('Could not run Kruskal-Wallis (scipy missing?). To install: pip install scipy')
    print('Error:', e)

## Key Observations (3 bullets)
- Observation 1: Replace this after running the notebook (e.g., 'Country X shows highest median GHI').
- Observation 2: Replace with a second insight (e.g., 'Country Y shows low variability in DNI').
- Observation 3: Replace with a third insight (e.g., 'Significant differences observed between countries for GHI (p < 0.05)').

In [ ]:
# Bonus: bar chart ranking countries by average GHI
if 'GHI' in metrics:
    avg = df_all.groupby('country')['GHI'].mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(6,4))
    sns.barplot(x=avg.index, y=avg.values, palette='viridis', ax=ax)
    ax.set_ylabel('Average GHI')
    ax.set_title('Average GHI by country')
    plt.show()
else:
    print('GHI not available to rank')